# Notebook 6 — Export & Packaging

**Project:** Robson Classification and C-Section Predictive System
**Stage:** Phase 5 — Preparing outputs for the dashboard, documentation, and reproducibility
**Scripts covered (6):**

| Script | Role | Status |
|--------|------|--------|
| `export_for_powerbi.py` | First export — 14 columns | **Superseded** by v2 |
| `export_for_powerbi_v2.py` | Expanded export — 29+ columns with pre-computed bands | **Current** |
| `step16_package.py` | Model outputs as flat tables + MODEL_CARD.md | **Current** |
| `export_csvs.py` | Cross-tabulated CSVs (group × payer, group × time, etc.) | **Current** |
| `make_csv.py` | Early CSV export | **Superseded** by step16 |
| `organise_project.py` | Folder structure + README.md + requirements.txt | **Current** |

---

### What this notebook documents

The transition from **analysis complete** to **deliverables ready**. Every number the dashboard shows, every table the report cites, and every file a reviewer would need to reproduce the project — all of it is produced by scripts in this notebook.

### The evolution story

| Version | What happened | Why it changed |
|---------|--------------|----------------|
| `make_csv.py` | First attempt: full CSV + sample + report table | CSV was 800+ MB, Excel truncated it silently |
| `export_for_powerbi.py` | Cut to 14 columns for Power BI | Cut too hard — dashboard couldn't compute anything |
| `export_for_powerbi_v2.py` | Expanded to 29+ columns with bands | Bands computed HERE (not DAX) to prevent drift |
| `step16_package.py` | Model outputs + MODEL_CARD.md | Two proposal commitments reversed, both documented |

---

## Part 1 — `export_for_powerbi.py` (v1, superseded)

### Why it was replaced

The 14-column export left the dashboard able to **display** results but not to **compute** anything. Without maternal age, parity, and comorbidity, the dashboard could not reproduce the standardisation, slice the Robson table by clinical variables, or build the VBAC page. DAX was ornamental rather than load-bearing.

### What it did right

The key insight is preserved in v2: **the export is itself a validation step.** When Power BI recomputes the CS rate in DAX and gets 32.35%, that is an independent check. Two different tools, same data, same answer — or a bug is found.

In [ ]:
"""
=============================================================================
EXPORT FOR POWER BI
=============================================================================
    python export_for_powerbi.py

Writes powerbi/births.csv — one row per classified birth, both years, with
only the columns the dashboard needs.

WHY THE FULL FILE AND NOT A SAMPLE

Power BI handles 7 million rows without difficulty; the reason people
sample is usually Excel, not Power BI. Using the full file matters here for
a specific reason: when the dashboard recomputes the caesarean rate in DAX
and gets 32.35%, that is the same figure the Python pipeline produced,
independently, in a different tool. If it came out slightly different
because the dashboard was built on a sample, you would have two numbers for
the same thing in one project — which is worse than not using DAX at all.

So this is not just a data export. It is a second, independent check on the
whole pipeline, and it either agrees or it finds a bug.

WHY THESE COLUMNS AND NOT ALL 60

Everything here is either a Robson output, the target, or a practice
variable the dashboard displays. The clinical risk factors are left out:
they belong to the model, and the dashboard shows the model's results
rather than recomputing it.

TARGETS TO CHECK AFTER LOADING
    overall caesarean rate 2023 .... 32.35%
    NTSV (groups 1, 2a, 2b) 2023 ... 26.56%
    groups 2b and 4b ............... exactly 100%
    unclassifiable 2023 ............ 0.55%
=============================================================================
"""

import pandas as pd
from pathlib import Path

OUT = Path("powerbi"); OUT.mkdir(exist_ok=True)

SOURCES = [
    ("natality_2023_extended.parquet", 2023),
    ("natality_2024_classified.parquet", 2024),
]

KEEP = ["Robson_group", "Robson_group_sort_order", "Classification_status",
        "Missing_variable", "is_cesarean", "payer", "weekend", "time_of_day",
        "APGAR5", "AB_NICU", "near_miss_flag"]

frames = []
for path, year in SOURCES:
    if not Path(path).exists():
        print(f"[skip] {path} not found")
        continue
    print(f"Reading {path} ...")
    d = pd.read_parquet(path)
    cols = [c for c in KEEP if c in d.columns]
    missing = [c for c in KEEP if c not in d.columns]
    if missing:
        print(f"  note: not present in this file — {missing}")
    s = d[cols].copy()
    s["year"] = year

    # tidy the two outcome columns into something a visual can use directly
    if "APGAR5" in s.columns:
        a = pd.to_numeric(s["APGAR5"], errors="coerce")
        s["apgar5_below7"] = (a < 7).where(a.notna() & (a <= 10)).astype("float")
        s = s.drop(columns=["APGAR5"])
    if "AB_NICU" in s.columns:
        s["nicu_admission"] = (s["AB_NICU"].astype(str).str.strip() == "Y").astype("int8")
        s = s.drop(columns=["AB_NICU"])

    s["is_classified"] = (s["Classification_status"] == "Classified").astype("int8")
    frames.append(s)
    print(f"  {len(s):,} rows\n")

if not frames:
    raise SystemExit("No source files found. Run the pipeline first.")

df = pd.concat(frames, ignore_index=True)

# a compact grouping label for the pages that report ten groups rather than subgroups
df["robson_group_10"] = df["Robson_group"].replace(
    {"2a": "2", "2b": "2", "4a": "4", "4b": "4", "5.1": "5", "5.2": "5"})

out = OUT / "births.csv"
print(f"Writing {out} ...")
df.to_csv(out, index=False, encoding="utf-8-sig")
mb = out.stat().st_size / 1e6
print(f"  {len(df):,} rows, {len(df.columns)} columns, {mb:,.0f} MB\n")

# ---- the numbers the dashboard must reproduce -----------------------------
print("=" * 66)
print("CHECK THESE AFTER LOADING INTO POWER BI")
print("=" * 66)
for year in sorted(df["year"].unique()):
    y = df[df.year == year]
    c = y[y.is_classified == 1]
    ntsv = c[c["Robson_group"].isin(["1", "2a", "2b"])]
    print(f"\n  {year}")
    print(f"    births loaded        {len(y):>12,}")
    print(f"    classified           {len(c):>12,}")
    print(f"    unclassifiable       {100*(1-len(c)/len(y)):>11.2f}%")
    print(f"    caesarean rate       {100*c.is_cesarean.mean():>11.2f}%")
    print(f"    NTSV rate            {100*ntsv.is_cesarean.mean():>11.2f}%")
    for g in ("2b", "4b"):
        sub = c[c["Robson_group"] == g]
        if len(sub):
            print(f"    group {g:<3}            {100*sub.is_cesarean.mean():>11.2f}%")

print("""
If Power BI reproduces these to two decimal places, the dashboard and the
Python pipeline agree and either can be cited. If it does not, stop and find
out why before building anything on top — a mismatch here means one of them
is wrong, and it is worth knowing which.
""")

---

## Part 2 — `export_for_powerbi_v2.py` (current)

### What changed and why

The export now carries every column the dashboard can genuinely use, both raw AND banded. The raw value lets DAX compute; the band gives clean slicer values that match the Python analysis exactly.

### Why bands are computed HERE, not in DAX

If age bands were defined again in DAX they could drift from the ones used in Stages 3 and 4. Two versions of "under 20" in one project is the kind of quiet inconsistency nobody notices until an examiner does.

### What is still left out

Model outputs (predicted probability, SHAP values). Those come from summary tables via `step16_package.py`, because recomputing a fitted model in DAX is neither possible nor desirable.

### New columns unlocked

| Columns | Dashboard page |
|---------|---------------|
| `age_band`, `bmi_band`, `parity_band`, `gestation_band` | Clinical slicers, standardisation |
| `any_diabetes`, `any_hypertension`, `any_comorbidity` | What indications actually explain |
| `delivery_detail`, `tolac_attempted`, `vbac_success` | VBAC page |
| `setting`, `payer`, `attendant` | Practice patterns |
| `presentation`, `DPLURAL`, `OEGest_Comb` | Show how classification was reached |

In [ ]:
"""
=============================================================================
EXPORT FOR POWER BI — EXPANDED
=============================================================================
    python export_for_powerbi_v2.py

Replaces the earlier 14-column export. That one was cut too hard: it left
the dashboard able to display results but not to compute anything, which
made DAX ornamental rather than load-bearing.

WHAT CHANGED AND WHY

The old rule was "the dashboard shows the model's results rather than
recomputing the model". That is right about the MODEL and wrong about
everything else. Without maternal age, parity and comorbidity the dashboard
cannot:

  - reproduce the standardisation, which is the project's central method
  - let anyone slice the Robson table by anything clinical
  - support a what-if parameter on the group 5 share
  - build the VBAC page

So the export now carries every column the dashboard can genuinely use,
raw AND banded. The raw value lets DAX compute; the band gives clean slicer
values that match the bands used in the Python analysis exactly, so a
figure produced in Power BI and a figure produced in the pipeline are
comparable rather than merely similar.

WHAT IS STILL LEFT OUT

Model outputs — predicted probability, SHAP values, performance metrics.
Those come from summary tables, because recomputing a fitted model in DAX
is neither possible nor desirable, and the model card restricts how its
output may be used.

BANDS ARE COMPUTED HERE, NOT IN DAX

Deliberately. If the age bands were defined again in DAX they could drift
from the ones used in stage 3 and stage 4, and two versions of "under 20"
in one project is the kind of quiet inconsistency nobody notices until an
examiner does.
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path

OUT = Path("powerbi"); OUT.mkdir(exist_ok=True)

SOURCES = [
    ("natality_2023_extended.parquet", 2023),
    ("natality_2024_classified.parquet", 2024),
]

# raw columns to carry through, grouped by what they enable in the dashboard
# Every column here has a page that uses it. Ten were dropped after review
# because they had no page: height, pre-pregnancy weight, weight gain,
# education, marital status, antenatal timing, smoking, fertility treatment,
# ART, previous preterm birth, interpregnancy interval and number of previous
# caesareans. Most were adjustment variables in the model, not dashboard
# content, and a field list carrying a third of its columns unused reads as
# undisciplined.
WANT = [
    # Robson classification and data quality           -> pages 2, 8
    "Robson_group", "Robson_group_sort_order", "Classification_status",
    "Missing_variable",
    # outcome                                          -> everywhere
    "is_cesarean", "RDMETH_REC", "ME_TRIAL", "RF_CESAR",
    # the six Robson criteria, so a viewer can see how the group was reached
    "ME_PRES", "DPLURAL", "OEGest_Comb", "LD_INDL", "PRIORLIVE", "PRIORDEAD",
    # maternal characteristics for WITHIN-group analysis -> page 6
    "MAGER", "BMI",
    # comorbidity, kept separate on purpose             -> page 6
    "RF_PDIAB", "RF_GDIAB", "RF_PHYPE", "RF_GHYPE", "RF_EHYPE",
    # practice setting                                  -> page 6
    "payer", "attendant", "BFACIL", "weekend", "time_of_day",
    # outcomes                                          -> page 5
    "APGAR5", "AB_NICU", "near_miss_flag",
]

SENTINELS = {
    "MAGER": [], "BMI": [99.9], "PRIORLIVE": [99], "PRIORDEAD": [99],
    "OEGest_Comb": [99], "DPLURAL": [9], "APGAR5": [99],
}
YN = ["RF_PDIAB", "RF_GDIAB", "RF_PHYPE", "RF_GHYPE", "RF_EHYPE",
      "RF_CESAR", "ME_TRIAL", "LD_INDL"]

frames = []
for path, year in SOURCES:
    if not Path(path).exists():
        print(f"[skip] {path} not found")
        continue
    print(f"Reading {path} ...")
    d = pd.read_parquet(path)
    have = [c for c in WANT if c in d.columns]
    missing = [c for c in WANT if c not in d.columns]
    if missing:
        print(f"  absent from this file: {missing}")
    s = d[have].copy()
    s["year"] = year
    frames.append(s)
    print(f"  {len(s):,} rows, {len(have)} columns kept\n")

if not frames:
    raise SystemExit("No source files found.")

df = pd.concat(frames, ignore_index=True)
print(f"Combined: {len(df):,} rows\n")

# ---------------------------------------------------------------------------
print("Cleaning sentinel codes ...")
for col, codes in SENTINELS.items():
    if col not in df.columns:
        continue
    x = pd.to_numeric(df[col], errors="coerce")
    if codes:
        n = int(x.isin(codes).sum())
        x = x.where(~x.isin(codes))
        if n:
            print(f"  {col:<14} {n:>10,} unknown codes removed")
    df[col] = x

for col in YN:
    if col not in df.columns:
        continue
    s = df[col].astype(str).str.strip().str.upper()
    v = pd.Series(np.nan, index=df.index)
    v[s == "Y"] = 1.0
    v[s == "N"] = 0.0
    df[col] = v

# ---------------------------------------------------------------------------
print("\nBuilding bands (same cut points as the Python analysis) ...")

df["age_band"] = pd.cut(df["MAGER"], [0, 19, 24, 29, 34, 99],
                        labels=["<20", "20-24", "25-29", "30-34", "35+"])
df["age_band_sort"] = df["age_band"].cat.codes

df["bmi_band"] = pd.cut(df["BMI"], [0, 18.5, 25, 30, 35, 99],
                        labels=["Underweight", "Normal", "Overweight",
                                "Obese I", "Obese II+"])
df["bmi_band_sort"] = df["bmi_band"].cat.codes

df["parity_band"] = pd.cut(df["PRIORLIVE"], [-1, 0, 1, 3, 99],
                           labels=["First birth", "Second", "3rd-4th", "5th+"])
df["parity_band_sort"] = df["parity_band"].cat.codes

df["gestation_band"] = pd.cut(df["OEGest_Comb"], [0, 33, 36, 38, 40, 50],
                              labels=["<34", "34-36", "37-38", "39-40", "41+"])
df["gestation_band_sort"] = df["gestation_band"].cat.codes

df["any_diabetes"] = ((df.get("RF_PDIAB", 0) == 1) |
                      (df.get("RF_GDIAB", 0) == 1)).astype("int8")
df["any_hypertension"] = ((df.get("RF_PHYPE", 0) == 1) |
                          (df.get("RF_GHYPE", 0) == 1) |
                          (df.get("RF_EHYPE", 0) == 1)).astype("int8")
df["any_comorbidity"] = ((df["any_diabetes"] == 1) |
                         (df["any_hypertension"] == 1)).astype("int8")

# readable labels for slicers
df["presentation"] = df["ME_PRES"].astype(str).str.strip().map(
    {"1": "Cephalic", "2": "Breech", "3": "Other"}).fillna("Unknown")

bf = df["BFACIL"].astype(str).str.strip() if "BFACIL" in df.columns else pd.Series("", index=df.index)
df["setting"] = np.select(
    [bf == "1", bf == "2", bf.isin(["3", "4", "5"]), bf.isin(["6", "7"])],
    ["Hospital", "Birth centre", "Home", "Clinic or other"], default="Unknown")

rd = df["RDMETH_REC"].astype(str).str.strip() if "RDMETH_REC" in df.columns else pd.Series("", index=df.index)
df["delivery_detail"] = np.select(
    [rd == "1", rd == "2", rd == "3", rd == "4", rd.isin(["5", "6"])],
    ["Vaginal", "VBAC", "Primary caesarean", "Repeat caesarean", "Vaginal after caesarean, other"],
    default="Unknown")

# TOLAC: achieved VBAC, or a repeat caesarean after a recorded trial of labour
df["tolac_attempted"] = np.where(
    (df.get("RF_CESAR", 0) == 1) & rd.isin(["2", "4"]),
    ((rd == "2") | ((rd == "4") & (df.get("ME_TRIAL", 0) == 1))).astype(float),
    np.nan)
df["vbac_success"] = np.where(
    (df.get("RF_CESAR", 0) == 1) & rd.isin(["2", "4"]),
    (rd == "2").astype(float), np.nan)

df["apgar5_below7"] = (df["APGAR5"] < 7).astype("float")
df["nicu_admission"] = (df["AB_NICU"].astype(str).str.strip() == "Y").astype("int8")
df["is_classified"] = (df["Classification_status"] == "Classified").astype("int8")
df["robson_group_10"] = df["Robson_group"].replace(
    {"2a": "2", "2b": "2", "4a": "4", "4b": "4", "5.1": "5", "5.2": "5"})

df = df.drop(columns=[c for c in ["ME_PRES", "BFACIL", "RDMETH_REC", "APGAR5",
                                  "AB_NICU"] if c in df.columns])

for c in df.select_dtypes("category").columns:
    df[c] = df[c].astype(str).replace("nan", "Unknown")

out = OUT / "births.parquet"
df.to_parquet(out, index=False)
mb = out.stat().st_size / 1e6

print(f"\nWritten {out}")
print(f"  {len(df):,} rows, {len(df.columns)} columns, {mb:,.1f} MB\n")

print("=" * 66)
print("REPRODUCE THESE IN POWER BI")
print("=" * 66)
for year in sorted(df["year"].unique()):
    y = df[df.year == year]
    c = y[y.is_classified == 1]
    n = c[c["Robson_group"].isin(["1", "2a", "2b"])]
    print(f"\n  {year}   classified {len(c):>10,}   "
          f"CS {100*c.is_cesarean.mean():.2f}%   NTSV {100*n.is_cesarean.mean():.2f}%")

print("\n" + "=" * 66)
print("WHAT THE NEW COLUMNS UNLOCK")
print("=" * 66)
print("""
  age_band, bmi_band, parity_band, gestation_band
      slice the Robson table by anything clinical, and drive the
      standardisation measure

  any_diabetes, any_hypertension, any_comorbidity
      a page on what the recorded indications actually explain

  delivery_detail, tolac_attempted, vbac_success
      the VBAC page: how often a trial of labour is offered, and how
      often it succeeds

  setting, payer, attendant
      practice patterns, with the caveat that attendant is unreliable for
      caesareans by NCHS's own documentation

  presentation, DPLURAL, OEGest_Comb
      the Robson criteria themselves, so a viewer can see how the
      classification was reached rather than taking it on trust
""")

---

## Part 3 — `step16_package.py`: Model Outputs + Model Card

### What gets written

| File | Contents |
|------|----------|
| `dash_model_performance.csv` | Headline metrics, validation vs test |
| `dash_calibration.csv` | Predicted vs observed by risk decile |
| `dash_by_group.csv` | Performance inside each Robson group |
| `dash_feature_importance.csv` | SHAP both rankings (mean |SHAP| and variance) |
| `dash_thresholds.csv` | Operating points and their cost in cases |
| **`MODEL_CARD.md`** | **What the model is for, what it is NOT for, and why** |

### The Model Card

This is the piece that matters most. It documents:
- Training population and evaluation metrics
- The distinction: predicts "performed" not "needed"
- **Approved uses:** aggregate estimation, SHAP explanation
- **Prohibited uses:** individual clinical decisions, applying to Egyptian case-mix
- **Two reversed commitments:** class-weighting removed, F2 threshold withdrawn — both with evidence

In [ ]:
"""
=============================================================================
STEP 16 — PACKAGE THE OUTPUTS
=============================================================================
    python step16_package.py

WHAT THIS IS FOR

Everything the model produced is currently spread across JSON files, console
output and a saved booster. Power BI cannot read any of that, and neither can
a reader six months from now.

This step turns it into flat tables that can be loaded directly, and — more
importantly — writes a model card recording what the model is for, what it
is not for, and what was decided along the way. Two of those decisions
reversed things stated in the proposal, and a project that changes course
without a written record of why is indistinguishable from one that never
noticed.

WHAT GETS WRITTEN

  dash_model_performance.csv    headline metrics, validation against test
  dash_calibration.csv          predicted against observed by risk decile
  dash_by_group.csv             performance inside each Robson group
  dash_feature_importance.csv   SHAP, both rankings, with the difference
                                between them made explicit
  dash_thresholds.csv           operating points and their cost in cases
  MODEL_CARD.md                 what it is for, what it is not for, and why
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
import json

need = ["test_results.json", "shap_importance.csv", "threshold_analysis.json"]
missing = [f for f in need if not Path(f).exists()]
if missing:
    raise SystemExit(f"Missing: {missing} — run steps 12 to 14 first.")

test = json.load(open("test_results.json"))
thr  = json.load(open("threshold_analysis.json"))
imp  = pd.read_csv("shap_importance.csv")
within = pd.read_csv("shap_within_group.csv") if Path("shap_within_group.csv").exists() else None

OUT = Path("dashboard_data"); OUT.mkdir(exist_ok=True)

# ---- 1. performance -------------------------------------------------------
v = test["validation_comparison"]
perf = pd.DataFrame([
    {"metric": "ROC-AUC (overall)",        "validation": v["roc_auc"],  "test": test["roc_auc"],
     "note": "inflated by separation BETWEEN groups"},
    {"metric": "ROC-AUC (within group)",   "validation": None,
     "test": f"{test['within_group_auc_range'][0]:.3f}-{test['within_group_auc_range'][1]:.3f}"
     if "within_group_auc_range" in test else None,
     "note": "the honest measure of skill"},
    {"metric": "AUPRC",                    "validation": v["auprc"],    "test": test["auprc"],
     "note": f"baseline {test['observed_rate']:.4f}"},
    {"metric": "Brier score",              "validation": v["brier"],    "test": test["brier"],
     "note": "lower is better"},
    {"metric": "Mean predicted rate",      "validation": v["mean_pred"],"test": test["mean_predicted"],
     "note": f"observed {test['observed_rate']:.4f}"},
    {"metric": "Calibration error (pp)",   "validation": None,          "test": test["calibration_mae_pp"],
     "note": "mean absolute, across risk deciles"},
])
perf.to_csv(OUT/"dash_model_performance.csv", index=False)

# ---- 2. calibration -------------------------------------------------------
if Path("test_calibration_curve.csv").exists():
    cal = pd.read_csv("test_calibration_curve.csv")
    cal.to_csv(OUT/"dash_calibration.csv", index=False)

# ---- 3. by group ----------------------------------------------------------
byg = pd.DataFrame(test["by_group"])
byg.to_csv(OUT/"dash_by_group.csv", index=False)

# ---- 4. feature importance, both rankings --------------------------------
imp = imp.rename(columns={"share_%": "overall_share_%"})
if within is not None:
    w = (within.groupby("feature")["sd_%"].mean()
                .rename("within_group_share_%").reset_index())
    imp = imp.merge(w, on="feature", how="left")
    imp["difference"] = (imp["overall_share_%"] - imp["within_group_share_%"]).round(2)
imp.to_csv(OUT/"dash_feature_importance.csv", index=False)

# ---- 5. thresholds --------------------------------------------------------
rows = []
for name, d in thr["operating_points"].items():
    rows.append({"criterion": name, "threshold": round(d["threshold"], 3),
                 "precision": round(d["precision"], 3), "recall": round(d["recall"], 3)})
pd.DataFrame(rows).to_csv(OUT/"dash_thresholds.csv", index=False)

# ---- 6. model card --------------------------------------------------------
wg = test.get("within_group_auc_range", [None, None])
card = f"""# Model Card

## What this model is

A gradient-boosted tree (XGBoost) predicting caesarean delivery from
characteristics known before the delivery decision, trained on the CDC
Natality Public Use File for 2023.

Training population: Robson groups 1, 2a, 3 and 4a — term, singleton,
cephalic, no uterine scar, labour occurred. {test['n_test']:,} held-out
births were used for the evaluation below.

## What it predicts, and what it does not

The target records whether a caesarean was PERFORMED, not whether one was
NEEDED. A model trained on outcomes learns the behaviour of the clinicians
who produced them. Because that behaviour is the phenomenon under study, it
cannot also serve as ground truth for necessity. Every use of this model
must carry that sentence with it.

## Performance on the held-out test set

| Metric | Value |
|---|---|
| ROC-AUC, overall | {test['roc_auc']:.4f} |
| ROC-AUC, within Robson group | {wg[0]:.3f} to {wg[1]:.3f} |
| AUPRC | {test['auprc']:.4f} (baseline {test['observed_rate']:.4f}) |
| Brier score | {test['brier']:.4f} |
| Mean predicted rate | {100*test['mean_predicted']:.2f}% against {100*test['observed_rate']:.2f}% observed |
| Calibration error | {test['calibration_mae_pp']:.2f} percentage points |

The overall figure is inflated by how easily the groups separate from one
another: group 3 runs at 2.5% and group 2a at 29%. Hold the group constant —
which is what a clinician actually knows — and discrimination falls to
{wg[0]:.3f}-{wg[1]:.3f}. That range is the honest description.

For comparison, the WHO C-Model, built on more than ten million deliveries
across 43 countries, reports AUC 0.832-0.844 and is used to compare
facilities rather than to predict for individuals. The ceiling here is a
property of the problem, not a shortfall in this implementation.

## Approved and prohibited uses

APPROVED — aggregate estimation. Summing predicted probabilities across a
population, and comparing facilities or groups. Calibration is
{test['calibration_mae_pp']:.2f} pp, which supports this.

APPROVED — explanation. SHAP analysis of what the recorded clinical picture
does and does not account for.

NOT APPROVED — individual clinical decisions. At the best available
operating point, precision is {thr['operating_points']['F0.5 (precision-weighted)']['precision']:.2f}: roughly six in ten flagged women do not
have a caesarean. The model must never be presented to a clinician as a
reason to book one. This mirrors ACOG guidance that a VBAC success score
must never become a barrier to trial of labour.

NOT APPROVED — applying this model to an Egyptian case mix to produce an
expected rate. The model over-predicts group 1 by 4.95 pp and under-predicts
group 2a by 5.54 pp. Those errors cancel on a US case mix and would not
cancel on Egypt's, where group 5 is three times larger. The Egypt expected
rate is computed from OBSERVED group-specific rates in the Robson table
instead.

## Decisions taken during development

**Class weighting removed.** scale_pos_weight of 7.56 corrected the 1:7.55
imbalance but inflated the mean predicted probability to 39.4% against 11.7%
observed, while leaving ROC-AUC unchanged at 0.7985. Removing it cut Brier
from 0.1857 to 0.0886 and calibration error from 27.66 pp to 0.11 pp.
Imbalance is instead handled through the choice of metric and threshold.

**Training population settled by test, not preference.** Narrow and wide
populations were both trained and scored on identical held-out rows.
Discrimination was indistinguishable (0.7985 against 0.7986), so the narrow
population — where clinical uncertainty actually lives — was chosen.

**F2 threshold withdrawn.** The proposal committed to maximising F2 on the
grounds that missing a needed caesarean is worse than a false alarm. That
reasoning holds for a diagnostic test. This model predicts what will be
done, not what is needed, and F2 would flag 38.6% of all births as high risk
in a country where 82.5% of caesarean decisions are already taken before
labour begins. No single threshold is adopted; the operating point is stated
per use.

**Three-family comparison replaced.** A single logistic-regression baseline
was used instead. XGBoost gained +0.0136 AUPRC over it, confirming
non-linearity without the selection pressure that comes from picking the
best of several models.

## What the model learned

Within a fixed Robson group, discrimination rests on maternal body size and
age far more than on documented medical risk factors:

| Group | Maternal body + age | Medical risk factors |
|---|---|---|
| 1 | 60.6% | 14.3% |
| 2a | 57.9% | 21.2% |
| 3 | 45.4% | 12.8% |
| 4a | 44.8% | 17.7% |

This is not a claim that medical conditions do not matter clinically. It is
a statement about prevalence and about what this data contains: gestational
diabetes affects 8% of these pregnancies and gestational hypertension 10%,
so documented indications cannot account for most caesareans among women
with no uterine scar at term with a single cephalic fetus.

## Limitations

- Cervical examination, Bishop score, membrane status, cardiotocography and
  estimated fetal weight are absent from birth certificate data. This is a
  pre-delivery risk model, not an admission-time decision model.
- Birth certificates are not medical records; several risk-factor fields are
  documented by NCHS as under-reported.
- Trained on one year (2023) in one country. Temporal validation on 2024 and
  geographic transfer to Egypt are both untested.
- Race and ethnicity are excluded by design, following the 2021 revision of
  the Grobman model.

## Reproducing this

Steps 4 to 14 in order. Random seed 42 throughout. The feature list is
generated from data_dictionary.py so that the reason for every inclusion and
exclusion stays attached to the decision.
"""
Path(OUT/"MODEL_CARD.md").write_text(card, encoding="utf-8")

print("Written to dashboard_data/:")
for f in sorted(OUT.iterdir()):
    print(f"  {f.name:<34} {f.stat().st_size/1024:>7.1f} KB")

print(f"""
The model card is the piece that matters most. Two decisions in it reverse
things the proposal committed to — dropping class weighting and withdrawing
the F2 threshold — and both are recorded with the evidence that forced the
change. A supervisor can check the reasoning; an examiner can see that the
objection was anticipated.

Next: temporal validation on 2024.
""")

---

## Part 4 — `export_csvs.py`: Cross-Tabulated Analysis Tables

Produces aggregate tables that require the 3.6M-row dataset to compute:

| File | Contents |
|------|----------|
| `A_group_by_payer.csv` | CS rate by Robson group × payer |
| `B_group_by_time_of_day.csv` | CS rate by group × time of day |
| `C_group_by_maternal_age.csv` | CS rate by group × age band |
| `D_missing_variable_counts.csv` | Data completeness report |
| `E_tolac_vbac_by_payer.csv` | TOLAC and VBAC rates by payer |
| `F_cdc_sample_200k.csv` | 200,000-row random sample (Excel-safe) |

In [ ]:
"""
=============================================================================
EXPORT TO CSV — run this on YOUR machine
=============================================================================

The nine CSVs supplied alongside this script contain every aggregate figure
the project has produced so far. They need no data and no setup.

This script produces the things that can only come from the 3.6 million
records sitting on your own disk. Run it in the same folder as
natality_2023_extended.parquet (created by stage4) or
natality_2023_filtered.parquet (created by robson_analysis.py).

    python export_csvs.py

Outputs go into a subfolder called csv_export/.

A note on the big one: 10_cdc_analysis_sample.csv is a 200,000-row random
sample, not the full file. Excel stops at 1,048,576 rows and would silently
truncate 3.6 million. If you want the whole thing for Power BI, uncomment
the Parquet line at the bottom — Power BI reads Parquet natively and it is
about twenty times smaller than the equivalent CSV.
"""

import pandas as pd
from pathlib import Path

OUT = Path("csv_export"); OUT.mkdir(exist_ok=True)

src = None
for candidate in ["natality_2023_extended.parquet", "natality_2023_filtered.parquet"]:
    if Path(candidate).exists():
        src = candidate; break
if src is None:
    raise SystemExit("No parquet cache found. Run robson_analysis.py first.")

print(f"Loading {src} ...")
df = pd.read_parquet(src)
print(f"{len(df):,} records\n")

d = df[df["Classification_status"] == "Classified"].copy()
d["apgar5_lt7"] = pd.to_numeric(d["APGAR5"], errors="coerce").lt(7)
d["nicu"] = d["AB_NICU"].astype(str).str.strip() == "Y"

# --- A. group x payer cross-tab -------------------------------------------
(d.groupby(["Robson_group_sort_order", "Robson_group", "payer"], observed=True)
   .agg(n=("is_cesarean", "size"), cs_rate_pct=("is_cesarean", "mean"))
   .reset_index().sort_values("Robson_group_sort_order")
   .drop(columns="Robson_group_sort_order")
   .assign(cs_rate_pct=lambda x: (100*x.cs_rate_pct).round(2))
   .to_csv(OUT/"A_group_by_payer.csv", index=False))

# --- B. group x time of day -----------------------------------------------
(d.groupby(["Robson_group_sort_order", "Robson_group", "time_of_day"], observed=True)
   .agg(n=("is_cesarean", "size"), cs_rate_pct=("is_cesarean", "mean"))
   .reset_index().sort_values("Robson_group_sort_order")
   .drop(columns="Robson_group_sort_order")
   .assign(cs_rate_pct=lambda x: (100*x.cs_rate_pct).round(2))
   .to_csv(OUT/"B_group_by_time_of_day.csv", index=False))

# --- C. maternal age band x group ------------------------------------------
age = pd.to_numeric(d["MAGER"], errors="coerce")
d["age_band"] = pd.cut(age, [0,19,24,29,34,39,99],
                       labels=["<20","20-24","25-29","30-34","35-39","40+"])
(d.groupby(["Robson_group_sort_order", "Robson_group", "age_band"], observed=True)
   .agg(n=("is_cesarean","size"), cs_rate_pct=("is_cesarean","mean"))
   .reset_index().sort_values("Robson_group_sort_order")
   .drop(columns="Robson_group_sort_order")
   .assign(cs_rate_pct=lambda x: (100*x.cs_rate_pct).round(2))
   .to_csv(OUT/"C_group_by_maternal_age.csv", index=False))

# --- D. data completeness --------------------------------------------------
mv = df["Missing_variable"].fillna("")
(mv[mv != ""].str.split(",").explode().value_counts()
   .rename_axis("missing_variable").reset_index(name="n_records")
   .assign(pct_of_all=lambda x: (100*x.n_records/len(df)).round(4))
   .to_csv(OUT/"D_missing_variable_counts.csv", index=False))

# --- E. VBAC / TOLAC -------------------------------------------------------
prev = df["RF_CESAR"].astype(str).str.strip() == "Y"
rdm  = df["RDMETH_REC"].astype(str).str.strip()
tri  = df["ME_TRIAL"].astype(str).str.strip()
v = df[prev & rdm.isin(["2","4"])].copy()
v["vbac"]  = rdm[prev & rdm.isin(["2","4"])] == "2"
v["tolac"] = v["vbac"] | ((rdm[prev & rdm.isin(["2","4"])]=="4") & (tri[prev & rdm.isin(["2","4"])]=="Y"))
(v.groupby("payer").agg(n=("tolac","size"), tolac_rate_pct=("tolac","mean"))
   .join(v[v.tolac].groupby("payer")["vbac"].mean().rename("vbac_success_pct"))
   .assign(tolac_rate_pct=lambda x:(100*x.tolac_rate_pct).round(2),
           vbac_success_pct=lambda x:(100*x.vbac_success_pct).round(2))
   .reset_index().to_csv(OUT/"E_tolac_vbac_by_payer.csv", index=False))

# --- F. analysis-ready sample ---------------------------------------------
keep = ["Robson_group","Classification_status","Missing_variable","is_cesarean",
        "MAGER","BMI","DPLURAL","ME_PRES","OEGest_Comb","DBWT","APGAR5",
        "RF_CESAR","RF_CESARN","LD_INDL","ME_TRIAL","ME_ROUT","RDMETH_REC",
        "RF_PDIAB","RF_GDIAB","RF_PHYPE","RF_GHYPE","RF_EHYPE",
        "payer","attendant","weekend","time_of_day","near_miss_flag","AB_NICU"]
keep = [c for c in keep if c in df.columns]
df[keep].sample(n=min(200_000, len(df)), random_state=42) \
        .to_csv(OUT/"F_cdc_sample_200k.csv", index=False)

# Full file for Power BI (Parquet — far smaller than CSV, and Power BI reads it):
# df[keep].to_parquet(OUT/"cdc_full_analysis_columns.parquet", index=False)

print("Written to csv_export/:")
for f in sorted(OUT.iterdir()):
    print(f"  {f.name:<34} {f.stat().st_size/1e6:>8.2f} MB")

---

## Part 5 — `make_csv.py` (superseded)

This was the first attempt at CSV export. It wrote three files: the full dataset (800+ MB CSV), an Excel-safe 200k sample, and the WHO report table.

**Superseded by** `step16_package.py` and `export_csvs.py`. Preserved in `archive/` as a record.

**The Excel trap it documented:** Excel stops at 1,048,576 rows and silently discards the rest. A 3.6M-row CSV opened in Excel shows only the first million rows with no warning.

In [ ]:
"""
=============================================================================
MAKE CSV — converts the analysis dataset to CSV
=============================================================================
Run:   python make_csv.py

Reads natality_2023_extended.parquet (the 69 MB file you already have) and
writes two CSVs plus the report table.

WHY NOT THE RAW .txt FILE
The raw file is fixed-width: columns are defined by character POSITION, not
by commas. Renaming it to .csv changes nothing — every row still opens as a
single squashed column. It also has no Robson classification in it. The
parquet already holds the 48 extracted columns plus the Robson group for
every one of the 3.6 million births, which is the actual analysis dataset.

THE EXCEL TRAP
Excel stops at 1,048,576 rows. Our data is 3.6 million. Opening the full
CSV in Excel silently keeps the first million rows and discards the rest,
with no warning at all. That is why this script writes a separate sampled
file that fits.
=============================================================================
"""

import pandas as pd
from pathlib import Path

SRC = "natality_2023_extended.parquet"
if not Path(SRC).exists():
    SRC = "natality_2023_filtered.parquet"
if not Path(SRC).exists():
    raise SystemExit("No parquet file found. Run this in the Nat2023us folder.")

print(f"Loading {SRC} ...")
df = pd.read_parquet(SRC)
print(f"  {len(df):,} rows, {len(df.columns)} columns\n")

COLS = [
    "Robson_group", "Classification_status", "Missing_variable", "is_cesarean",
    "MAGER", "BMI", "M_Ht_In", "PWgt_R",
    "PRIORLIVE", "PRIORDEAD", "RF_CESAR", "RF_CESARN",
    "LD_INDL", "LD_AUGM", "ME_TRIAL", "ME_PRES", "ME_ROUT",
    "RDMETH_REC", "DMETH_REC", "DPLURAL", "OEGest_Comb", "DBWT",
    "APGAR5", "AB_NICU", "near_miss_flag",
    "RF_PDIAB", "RF_GDIAB", "RF_PHYPE", "RF_GHYPE", "RF_EHYPE",
    "payer", "attendant", "weekend", "time_of_day",
]
COLS = [c for c in COLS if c in df.columns]
out = df

# ---- 1. Full dataset, written in chunks so memory stays flat -------------
print("Writing robson_full_dataset.csv (this is the big one) ...")
first = True
for i in range(0, len(out), 500_000):
    out.iloc[i:i+500_000].to_csv(
        "robson_full_dataset.csv", mode="w" if first else "a",
        header=first, index=False)
    first = False
    print(f"  {min(i+500_000, len(out)):,} / {len(out):,}")
mb = Path("robson_full_dataset.csv").stat().st_size / 1e6
print(f"  done — {mb:,.0f} MB\n")

# ---- 2. Excel-safe random sample ------------------------------------------
print("Writing robson_sample_200k.csv ...")
out.sample(n=min(200_000, len(out)), random_state=42) \
   .to_csv("robson_sample_200k.csv", index=False)
print(f"  done — {Path('robson_sample_200k.csv').stat().st_size/1e6:.0f} MB\n")

# ---- 3. The WHO report table ----------------------------------------------
print("Writing robson_report_table.csv ...")
d = df[df["Classification_status"] == "Classified"]
N, C = len(d), int(d["is_cesarean"].sum())
t = (d.groupby(["Robson_group_sort_order", "Robson_group"], observed=True)
       .agg(n_women=("is_cesarean", "size"), n_cesarean=("is_cesarean", "sum"))
       .reset_index().sort_values("Robson_group_sort_order")
       .drop(columns="Robson_group_sort_order"))
t["group_size_pct"]             = (100 * t.n_women / N).round(2)
t["group_cs_rate_pct"]          = (100 * t.n_cesarean / t.n_women).round(2)
t["absolute_contribution_pct"]  = (100 * t.n_cesarean / N).round(2)
t["relative_contribution_pct"]  = (100 * t.n_cesarean / C).round(2)
t.loc[len(t)] = ["TOTAL", N, C, 100.0, round(100*C/N, 2), round(100*C/N, 2), 100.0]
t.to_csv("robson_report_table.csv", index=False)
print("  done\n")

print("=" * 60)
print("FILES WRITTEN")
print("=" * 60)
print("  robson_report_table.csv    <- show this to your supervisor first")
print("  robson_sample_200k.csv     <- opens in Excel, safe")
print(f"  robson_full_dataset.csv    <- {mb:,.0f} MB, DO NOT open in Excel")
print("\nExcel truncates at 1,048,576 rows without warning.")
print("For the full file use Power BI, or read it in Python.")

---

## Part 6 — `organise_project.py`: The Final Structure

### Why this matters more than it looks

The scripts are not workings-out to be cleared away. They ARE the answer: every figure exists because a named script produced it from a named input. A folder with 29 scripts in no order is functionally the same as no scripts at all.

### The structure

| Folder | Contents |
|--------|----------|
| `01_classifier/` | Robson engine + tests + kappa audit |
| `02_reference/` | CDC pipeline, validation, confounding, temporal |
| `03_model/` | Steps 4-16 in execution order |
| `04_egypt/` | DHS, EHIS, standardisation, gap, cost, governorates |
| `05_outputs/` | Tables, figures, MODEL_CARD.md |
| `06_data/` | Large intermediate files (not for sharing) |
| `archive/` | Superseded scripts (step10 v1, find_cs_var, make_csv) |

### What it also produces

- **`README.md`** — execution order with the reasoning for every key decision
- **`requirements.txt`** — with xgboost pinned to 2.1.4 (3.4.0 crashes silently on Python 3.14)

### The strongest sentence in the project

> "Every figure in this project is produced by a named script from a named input, and the README gives the order."

The difference between a student project and a research project is not the numbers — it is that someone else can reach the same numbers.

In [ ]:
"""
=============================================================================
ORGANISE THE PROJECT
=============================================================================
    python organise_project.py            # show what it would do
    python organise_project.py --apply    # actually move the files

WHY THIS MATTERS MORE THAN IT LOOKS

The scripts are not workings-out to be cleared away once the answer is
found. They ARE the answer: every figure in this project exists because a
named script produced it from a named input, and a reader who cannot follow
that chain has to take the numbers on trust. A folder with twenty-nine
scripts in no order is functionally the same as no scripts at all.

Nothing is deleted. Files are moved into folders that say what they are for,
and a README is written that names the order things run in.

STRUCTURE

  01_classifier/     the Robson rule engine and its tests
  02_reference/      building and validating the CDC reference population
  03_model/          the predictive model, in the order the steps run
  04_egypt/          Egyptian sources, standardisation, cost, governorates
  05_outputs/        tables and figures meant to be read
  06_data/           large intermediate files (not for sharing)
  archive/           superseded or one-off scripts, kept for the record
=============================================================================
"""

import shutil, sys
from pathlib import Path

APPLY = "--apply" in sys.argv

README = """# Robson Classification and Machine-Learning Prediction of Caesarean Delivery

Every figure in this project is produced by a named script from a named
input. This file gives the order.

## Setup

    pip install -r requirements.txt

xgboost is pinned to 2.1.4 deliberately: version 3.4.0 crashes silently on
Python 3.14 — it imports and then dies on the first training call with no
error message.

## Data you need

Neither file is included; both are public.

  CDC Natality Public Use File, 2023 and 2024 (~4.9 GB each)
      https://www.cdc.gov/nchs/data_access/VitalStatsOnline.htm
  Egypt DHS 2014 and Health Issues Survey 2015 Individual Recodes
      https://dhsprogram.com/data/  (free, registration and a stated
      research purpose required)

Put them in 06_data/.

## Order of execution

### 1. Classifier — 01_classifier/

    python test_robson.py

21 hand-written cases, one or more per Robson group, each annotated in
advance with the group it must produce. Run this before anything else; it
needs no data and it catches rule errors that would otherwise propagate
silently into every result.

kappa_audit.py generates a blinded worksheet for manual verification against
the WHO flowchart. Run it when you are ready to do that check by hand.

### 2. Reference population — 02_reference/

    python cdc_robson_pipeline.py Nat2023PublicUS.txt
    python robson_analysis.py
    python stage3_confounding.py
    python stage4_setting_and_wider_adjustment.py Nat2023PublicUS.txt
    python temporal_validation_2024.py Nat2024PublicUS.txt

The second script contains the stop gate: derived figures are checked
against statistics NCHS publishes independently from the same birth
certificates. Overall caesarean rate 32.35% against a published 32.3%, and
the low-risk NTSV rate 26.56% against roughly 26%. NCHS defines its low-risk
measure as exactly Robson groups 1 and 2 combined, so reproducing it
confirms the classification rather than merely agreeing with it.

Do not proceed past that check if it fails.

### 3. Model — 03_model/

    python data_dictionary.py
    python step4_5_clean.py
    python step6_encode.py
    python step7_9_split.py
    python step10_train_v2.py
    python step11b_calibration.py
    python step12_threshold.py
    python step13_evaluate.py
    python step14_shap.py
    python step14b_shap_within.py
    python step16_package.py

Two steps here reverse decisions the proposal committed to, and both are
documented with the evidence that forced the change:

  step11b  class weighting removed. scale_pos_weight of 7.56 left ROC-AUC
           unchanged but inflated mean predicted probability from 11.7% to
           39.4%. Calibration error fell from 27.66 to 0.11 percentage
           points once it was dropped.
  step12   the F2 threshold was withdrawn. F2 flags 38.6% of all births as
           high risk with 76% false alarms, which in Egypt would manufacture
           prediction rather than catch need.

step13 opens the test set. It is opened once. If the result disappoints, it
is reported — re-tuning after looking would turn it into a second
validation set and every later number would be quietly optimistic.

### 4. Egypt — 04_egypt/

    python dhs_egypt.py
    python ehis_2015.py
    python egypt_standardisation.py
    python egypt_gap.py
    python egypt_governorates.py
    python egypt_cost.py

egypt_standardisation.py imports its standardisation function from
stage3_confounding.py rather than reimplementing it. The two countries are
analysed by the same function, not by similar methods, and that is the point
of the comparison.

egypt_gap.py is a tool: add a case mix to the EGYPT dictionary and every
table recomputes.

## Where results live

  05_outputs/       tables, figures, MODEL_CARD.md
  06_data/          large intermediates, not for sharing

MODEL_CARD.md states what the model may and may not be used for. The
prohibition that matters most: it must not be applied to an Egyptian case
mix to produce an expected rate. It over-predicts group 1 by 4.95 points and
under-predicts group 2a by 5.54; those errors cancel on a US case mix and
would not cancel on Egypt's, where group 5 is three times larger. The Egypt
expected rate is computed from observed group-specific rates instead.

## Reproducibility

Random seed 42 throughout. The feature list is generated from
data_dictionary.py so the reason for every inclusion and exclusion stays
attached to the decision rather than living in someone's memory.
"""


PLAN = {
    "01_classifier": [
        "robson_classifier.py",
        "test_robson.py",
        "kappa_audit.py",
    ],
    "02_reference": [
        "cdc_robson_pipeline.py",
        "robson_analysis.py",
        "stage3_confounding.py",
        "stage4_setting_and_wider_adjustment.py",
        "temporal_validation_2024.py",
    ],
    "03_model": [
        "data_dictionary.py",
        "model_features.py",
        "step4_5_clean.py",
        "step6_encode.py",
        "step7_9_split.py",
        "step10_train_v2.py",
        "step11b_calibration.py",
        "step12_threshold.py",
        "step13_evaluate.py",
        "step14_shap.py",
        "step14b_shap_within.py",
        "step16_package.py",
    ],
    "04_egypt": [
        "dhs_egypt.py",
        "ehis_2015.py",
        "egypt_standardisation.py",
        "egypt_gap.py",
        "egypt_governorates.py",
        "egypt_cost.py",
    ],
    "archive": [
        "step10_train.py",        # superseded by v2 after the memory crash
        "find_cs_var.py",         # one-off: locating the EHIS caesarean variable
        "make_csv.py",            # superseded by step16_package.py
        "export_csvs.py",
    ],
}

# outputs by pattern rather than by name, since these accumulate
OUTPUT_PATTERNS = ["*.csv", "*.json", "*.png", "MODEL_CARD.md"]
DATA_PATTERNS   = ["*.parquet", "*.txt", "*.DTA", "*.dat"]
KEEP_AT_ROOT    = {"organise_project.py", "README.md", "requirements.txt"}

root = Path(".")
moves, missing = [], []

for folder, files in PLAN.items():
    for f in files:
        src = root / f
        if src.exists():
            moves.append((src, root / folder / f))
        else:
            missing.append(f)

for pat in OUTPUT_PATTERNS:
    for src in root.glob(pat):
        if src.name in KEEP_AT_ROOT or not src.is_file():
            continue
        moves.append((src, root / "05_outputs" / src.name))

for pat in DATA_PATTERNS:
    for src in root.glob(pat):
        if not src.is_file():
            continue
        moves.append((src, root / "06_data" / src.name))

print(f"{'MOVING' if APPLY else 'WOULD MOVE'} {len(moves)} files\n")
by_dest = {}
for s, d in moves:
    by_dest.setdefault(d.parent.name, []).append(s.name)
for folder in sorted(by_dest):
    print(f"  {folder}/  ({len(by_dest[folder])} files)")
    for n in sorted(by_dest[folder])[:6]:
        print(f"      {n}")
    if len(by_dest[folder]) > 6:
        print(f"      ... and {len(by_dest[folder])-6} more")
    print()

if missing:
    print(f"Not found (fine if you never downloaded them): {', '.join(missing)}\n")

if APPLY:
    for s, d in moves:
        d.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(s), str(d))
    print("Moved.\n")

    Path("requirements.txt").write_text(
        "pandas>=2.0\nnumpy>=1.24\npyarrow>=14.0\nxgboost==2.1.4\n"
        "scikit-learn>=1.3\nshap>=0.44\nmatplotlib>=3.7\npyreadstat>=1.2\n",
        encoding="utf-8")
    Path("README.md").write_text(README, encoding="utf-8")
    print("Wrote README.md and requirements.txt")
else:
    print("Nothing moved. Re-run with --apply to do it.\n")

---

## Summary — The Complete Notebook Set

All six notebooks are now complete:

| Notebook | Scripts | Cells | Focus |
|----------|---------|-------|-------|
| 01 — Reference & Classification | 4 | 38 | Robson engine, 21 tests, CDC pipeline, data dictionary |
| 02 — Descriptive Analysis | 2 + 9 CSVs | 34 | Validation, Group 9, VBAC, practice patterns, safety |
| 03 — Egypt Standardisation | 8 | 19 | DHS/EHIS, direct standardisation, **indirect standardisation (8-cat)**, cost, governorates |
| 04 — Predictive Model | 8 | 18 | Clean → encode → split → train → **calibration fix** → threshold → evaluate |
| 05 — Interpretation & Validation | 4 | 10 | SHAP (global + within-group), temporal 2024, kappa tool |
| 06 — Export & Packaging | 6 | 14 | Power BI export, model card, CSV tables, project organisation |

**Total:** 32 scripts documented across 133 cells.

### What comes next

```
✅ Notebooks 1-6 complete
⬅️ 00_MASTER.ipynb         ← reads CSVs, shows results, opens in seconds
   Word document (19 chapters)
   Committee preparation
```

---
*End of Notebook 6 — End of the documentation notebooks*